<a href="https://colab.research.google.com/github/Houry6/ds-ifri/blob/main/TP2_donn%C3%A9es_finances_graphique.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TP2: Données financières et graphique

Objectif: récupérer le cours boursier d'une action et ses revenus trimestriels
depuis deux sources différentes, nettoyer les données, puis les représenter
sur un même graphique interactif.

Action étudiée: Tesla (TSLA)
Outils: Python, Pandas, yfinance, requests, Plotly

1. Installation et imports
- Installation des bibliothèques nécessaires et import des outils :
- yfinance pour les cours, requests, pandas pour le scraping et le traitement et plotly pour le graphique interactif.

In [18]:

import yfinance as yf
import pandas as pd
import requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots

2. Collecte: historique du cours de l'action
Le cours de clôture de l'action sur 5 ans est récupéré avec la bibliothèque yfinance,
qui interroge Yahoo Finance. On obtient un DataFrame avec une ligne par jour de bourse.
On ne garde que la date et le cours de clôture, et on retire le fuseau horaire
pour pouvoir aligner les dates avec l'autre jeu de données.

In [19]:
ticker = "TSLA"
cours = yf.Ticker(ticker).history(period="5y").reset_index()
cours["Date"] = pd.to_datetime(cours["Date"]).dt.tz_localize(None)
cours = cours[["Date", "Close"]]
print(cours.shape)
cours.head()

(1254, 2)


,Date,Close
0,2021-10-18,290.036682
1,2021-10-19,288.089996
2,2021-10-20,288.600006
3,2021-10-21,298.000000
4,2021-10-22,303.226654


3. Scraping: revenus trimestriels
Les revenus trimestriels ne sont pas disponibles par API; ils sont publiés dans un
tableau HTML sur une page web. La page est téléchargée avec requests, après
vérification du code de statut (200 = succès). Le tableau est ensuite extrait avec
pd.read_html, qui transforme directement les balises <table> en DataFrame.

In [20]:
url = "https://stockanalysis.com/stocks/tsla/financials/?p=quarterly"

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

rep = requests.get(url, headers=headers)
print("Status:", rep.status_code)

tables = pd.read_html(rep.text)
brut = tables[0]
brut.head()

Status: 200


/tmp/ipykernel_1092/3515915935.py:8: FutureWarning:

Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.



,Fiscal Quarter,Q2 2026,Q1 2026,Q4 2025,Q3 2025,Q2 2025,Q1 2025,Q4 2024,Q3 2024,Q2 2024,...,Q4 2023,Q3 2023,Q2 2023,Q1 2023,Q4 2022,Q3 2022,Q2 2022,Q1 2022,Q4 2021,Q3 2021
,Period Ending,"Jun '26 Jun 30, 2026","Mar '26 Mar 31, 2026","Dec '25 Dec 31, 2025","Sep '25 Sep 30, 2025","Jun '25 Jun 30, 2025","Mar '25 Mar 31, 2025","Dec '24 Dec 31, 2024","Sep '24 Sep 30, 2024","Jun '24 Jun 30, 2024",...,"Dec '23 Dec 31, 2023","Sep '23 Sep 30, 2023","Jun '23 Jun 30, 2023","Mar '23 Mar 31, 2023","Dec '22 Dec 31, 2022","Sep '22 Sep 30, 2022","Jun '22 Jun 30, 2022","Mar '22 Mar 31, 2022","Dec '21 Dec 31, 2021","Sep '21 Sep 30, 2021"
0,Revenue Revenue Growth,28236,22387,24901,28095,22496,19335,25707,25182,25500,...,25167,23350,24927,23329,24318,21454,16934,18756,17719,13757
1,Revenue Growth,25.52%,15.78%,-3.14%,11.57%,-11.78%,-9.23%,2.15%,7.85%,2.30%,...,3.49%,8.84%,47.20%,24.38%,37.24%,55.95%,41.61%,80.54%,64.92%,56.85%
2,Gross Profit Gross Profit Growth,4751,4720,5009,5054,3878,3153,4179,4997,4578,...,4438,4178,4533,4511,5777,5382,4234,5460,4847,3660
3,Operating Income Operating Income Growth,398,941,1171,1862,923,493,1583,2772,2188,...,2064,1764,2399,2664,3865,3688,2500,3603,2613,2004
4,Net Income Net Income Growth,1116,477,840,1373,1172,409,2125,2173,1400,...,7927,1851,2703,2518,3722,3292,2256,3313,2326,1618


### 4. Nettoyage
Les données récupérées sont brutes, au format texte et disposées en colonnes pour chaque trimestre.

On isole simplement la ligne des revenus pour la remettre au propre : on vire le signe $, les virgules de séparation et on convertit le tout en nombres. Pareil pour les dates, on les nettoie pour obtenir un format propre avant de trier chronologiquement et d'enlever les éventuels trous.

In [21]:
# Sélection de la ligne Revenue
ligne_revenue = brut[brut.iloc[:, 0].str.contains("Revenue", na=False)].iloc[0]

# Transposition des dates en lignes
df_temp = ligne_revenue.iloc[1:].reset_index()

# Extraction des colonnes de date et de valeur
rev = df_temp[["level_1", 0]].copy()
rev.columns = ["Date", "Revenu"]

# Nettoyage des valeurs numériques
rev["Revenu"] = rev["Revenu"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False)
rev["Revenu"] = pd.to_numeric(rev["Revenu"], errors="coerce")

# Nettoyage et conversion des dates
rev["Date"] = rev["Date"].astype(str).str.split().str[-3:].str.join(" ")
rev["Date"] = pd.to_datetime(rev["Date"], errors="coerce")

# Tri chronologique et suppression des vides
rev = rev.dropna().sort_values("Date")
rev.head()

,Date,Revenu
19,2021-09-30,13757
18,2021-12-31,17719
17,2022-03-31,18756
16,2022-06-30,16934
15,2022-09-30,21454


5. Visualisation: cours et revenus sur un même graphique
Le cours (en dollars) et les revenus (en millions de dollars) n'ont pas la même échelle.
Le graphique utilise donc deux axes verticaux: le cours en courbe sur l'axe de gauche,
les revenus en barres sur l'axe de droite. Plotly rend le graphique interactif
(zoom, survol des valeurs).

In [22]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=cours["Date"], y=cours["Close"],
                         name="Cours de clôture"), secondary_y=False)
fig.add_trace(go.Bar(x=rev["Date"], y=rev["Revenu"],
                     name="Revenus trimestriels (M$)", opacity=0.4),
              secondary_y=True)
fig.update_layout(title=f"{ticker} : cours et revenus trimestriels")
fig.update_yaxes(title_text="Cours ($)", secondary_y=False)
fig.update_yaxes(title_text="Revenus (millions $)", secondary_y=True)
fig.show()

Sur la période étudiée, les revenus trimestriels de Tesla augmentent globalement
(d'environ 14 000 M à plus de 25 000 M), avec quelques trimestres en recul.
Le cours de l'action, lui, est beaucoup plus volatil: il passe d'environ 110 début 2023 à près de 490 fin 2025.

Les deux séries ne évoluent pas de façon parallèle. Par exemple, début 2023, les
revenus sont élevés alors que le cours est à son plus bas. Le cours semble donc
dépendre d'autres facteurs que les seuls revenus (anticipations, contexte de marché).
Ce graphique montre une tendance, mais ne permet pas de conclure à un lien de cause à effet.

Les données proviennent de sources gratuites, et le scraping dépend de la structure de la page web, qui peut changer.